# 🩺 Liver Disease Chatbot — LLM Fine-Tuning (Google Colab & Local PC)

This notebook fine-tunes **Phi-3-mini-4k-instruct** (3.8B parameters) on your liver disease Q&A dataset using **QLoRA** — 4-bit quantization + LoRA adapters.

It is fully configured to work **either** on **Google Colab** (with free T4 GPU or A100) **or** on your **Local PC** (supporting Windows & Linux GPU/CPU setups).

## Setup Prerequisites (If running locally):
1. **GPU Required**: A CUDA-compatible NVIDIA GPU (recommended 8GB+ VRAM for 4-bit QLoRA, e.g. RTX 3060/4060 or higher).
2. **Install CUDA**: Ensure CUDA Toolkit is installed (matching your PyTorch version).
3. **Windows `bitsandbytes`**: If training on Windows, install the windows-compatible port: 
   `pip install bitsandbytes --index-url https://jllllll.github.io/bitsandbytes-windows-webui`
4. Ensure your dataset is located at `Data/liver_qa_dataset.jsonl` in your project folder.

In [1]:
# ── Cell 1: Check GPU / Hardware ─────────────────────────────────────────────
import torch
import subprocess
import sys

print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
cuda_available = torch.cuda.is_available()
print(f"CUDA (GPU) Available: {cuda_available}")

if cuda_available:
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
    try:
        result = subprocess.run(['nvidia-smi'], capture_output=True, text=True, check=True)
        print(result.stdout)
    except Exception:
        print("CUDA is active, but nvidia-smi command is not available in system PATH.")
else:
    print("\n⚠️ No CUDA GPU detected. Training will run on CPU, which is extremely slow and NOT recommended.")

Python version: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
PyTorch version: 2.10.0+cpu
CUDA (GPU) Available: False

⚠️ No CUDA GPU detected. Training will run on CPU, which is extremely slow and NOT recommended.


In [2]:
# ── Cell 2: Install dependencies ──────────────────────────────────────────────
# Uncomment and run this cell if dependencies are missing.
# For Google Colab:
# !pip install -q transformers peft trl bitsandbytes datasets accelerate huggingface_hub

# For Local Windows (if you get bitsandbytes errors):
# pip install bitsandbytes --index-url https://jllllll.github.io/bitsandbytes-windows-webui

In [3]:
# ── Cell 3: Load dataset (Colab Upload vs Local Paths) ───────────────────────
import sys
import os

is_colab = 'google.colab' in sys.modules
dataset_path = None

if is_colab:
    from google.colab import files
    print("Running on Google Colab. Please upload your 'liver_qa_dataset.jsonl' file...")
    uploaded = files.upload()
    if uploaded:
        dataset_path = list(uploaded.keys())[0]
        print(f"Uploaded successfully: {dataset_path}")
else:
    print("Running locally. Searching for liver_qa_dataset.jsonl in workspace paths...")
    # Standard project directories to check relative to notebook
    candidate_paths = [
        '../Data/liver_qa_dataset.jsonl',
        'Data/liver_qa_dataset.jsonl',
        './liver_qa_dataset.jsonl',
        './Data/liver_qa_dataset.jsonl'
    ]
    for path in candidate_paths:
        if os.path.exists(path):
            dataset_path = path
            break
            
    if dataset_path:
        print(f"Found local dataset at: {os.path.abspath(dataset_path)}")
    else:
        dataset_path = input("Could not find dataset automatically. Please enter absolute or relative path to liver_qa_dataset.jsonl: ")
        if not os.path.exists(dataset_path):
            raise FileNotFoundError(f"Specified dataset path does not exist: {dataset_path}")

print(f"Active dataset path: {dataset_path}")

Running locally. Searching for liver_qa_dataset.jsonl in workspace paths...
Found local dataset at: f:\Liver Disease Detectioon system 2\Data\liver_qa_dataset.jsonl
Active dataset path: ../Data/liver_qa_dataset.jsonl


In [4]:
# ── Cell 4: Load and inspect dataset ─────────────────────────────────────────
import json

records = []
with open(dataset_path, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print(f"Total examples loaded: {len(records)}")
print("\nSample entry preview:")
for msg in records[0]['messages']:
    role = msg['role']
    preview = msg['content'][:150].replace('\n', ' ')
    print(f"  [{role}] {preview}...")

Total examples loaded: 341

Sample entry preview:
  [system] You are LiverAI — a friendly, easy-to-understand liver health assistant. Your goal is to help patients, caregivers, and students learn about liver dis...
  [user] What leads to Autoimmune Causes?...
  [assistant] Autoimmune Hepatitis (AIH): - Destruction of liver parenchyma by autoantibodies - More common in females - Lab: Elevated ANA, anti-smooth muscle antib...


In [5]:
# ── Cell 5: Configuration ─────────────────────────────────────────────────────
import sys

is_colab = 'google.colab' in sys.modules

# Set output directory locally inside the project's models folder, or Colab default
output_dir = '/content/liver-lora' if is_colab else '../models/liver-lora'

CONFIG = {
    'base_model':   'microsoft/Phi-3-mini-4k-instruct', # e.g. 'Qwen/Qwen2.5-1.5B-Instruct' for low-VRAM GPUs
    'output_dir':   output_dir,
    'epochs':        3,
    'batch_size':    2,
    'grad_accum':    4,       # Effective batch size = batch_size * grad_accum = 8
    'lr':            2e-4,
    'max_seq_len':   1024,
    'lora_r':        16,
    'lora_alpha':    32,
    'lora_dropout':  0.05,
    'val_split':     0.1,
}
print('Configuration set:', CONFIG)

Configuration set: {'base_model': 'microsoft/Phi-3-mini-4k-instruct', 'output_dir': '../models/liver-lora', 'epochs': 3, 'batch_size': 2, 'grad_accum': 4, 'lr': 0.0002, 'max_seq_len': 1024, 'lora_r': 16, 'lora_alpha': 32, 'lora_dropout': 0.05, 'val_split': 0.1}


In [9]:
# ── Cell 6: Load model + tokenizer with 4-bit QLoRA ──────────────────────────
import torch
import os
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 1. FIXED: Must be the folder path, NOT the filename
LOCAL_MODEL_PATH = r"F:\Liver Disease Detectioon system 2\notebooks" 

# 2. REQUIRED: Replace this string with your exact model repo name from Hugging Face
HF_MODEL_NAME = "unsloth/llama-3-8b-Instruct"

print(f"Reading weights locally from: {LOCAL_MODEL_PATH}")
print(f"Fetching minimal configuration templates from: {HF_MODEL_NAME}\n")

device = "cuda" if torch.cuda.is_available() else "cpu"
use_qlora = device == "cuda"
bnb_config = None

if use_qlora:
    try:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        print("BitsAndBytes config initialized for 4-bit QLoRA.")
    except Exception as e:
        print(f"Failed to load BitsAndBytes config: {e}. Falling back to standard float16 precision.")
        use_qlora = False

# Load tokenizer configs from online template safely
tokenizer = AutoTokenizer.from_pretrained(
    HF_MODEL_NAME,
    trust_remote_code=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model_kwargs = {
    "trust_remote_code": True,
}

if use_qlora and bnb_config:
    model_kwargs["quantization_config"] = bnb_config
    model_kwargs["device_map"] = "auto"
else:
    model_kwargs["device_map"] = "auto" if device == "cuda" else None
    if device == "cuda":
        model_kwargs["torch_dtype"] = torch.float16

# Load config profile from HF, but load raw layers from local path
model = AutoModelForCausalLM.from_pretrained(
    LOCAL_MODEL_PATH,             # Looks inside your folder for the two .safetensors shards
    config=HF_MODEL_NAME,         # Downloads tiny structural configuration schema online
    **model_kwargs
)
print(f"Model successfully loaded from disk onto target device: {device.upper()}!")


Reading weights locally from: F:\Liver Disease Detectioon system 2\notebooks
Fetching minimal configuration templates from: unsloth/llama-3-8b-Instruct



ValueError: Unrecognized model in F:\Liver Disease Detectioon system 2\notebooks. Should have a `model_type` key in its config.json, or contain one of the following strings in its name: aimv2, aimv2_vision_model, albert, align, altclip, apertus, arcee, aria, aria_text, audio-spectrogram-transformer, autoformer, aya_vision, bamba, bark, bart, beit, bert, bert-generation, big_bird, bigbird_pegasus, biogpt, bit, bitnet, blenderbot, blenderbot-small, blip, blip-2, blip_2_qformer, bloom, blt, bridgetower, bros, camembert, canine, chameleon, chinese_clip, chinese_clip_vision_model, clap, clip, clip_text_model, clip_vision_model, clipseg, clvp, code_llama, codegen, cohere, cohere2, cohere2_vision, colpali, colqwen2, conditional_detr, convbert, convnext, convnextv2, cpmant, csm, ctrl, cvt, d_fine, dab-detr, dac, data2vec-audio, data2vec-text, data2vec-vision, dbrx, deberta, deberta-v2, decision_transformer, deepseek_v2, deepseek_v3, deepseek_vl, deepseek_vl_hybrid, deformable_detr, deit, depth

In [ ]:
# ── Cell 7: Apply LoRA adapters ───────────────────────────────────────────────
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

if use_qlora:
    model = prepare_model_for_kbit_training(model)

# Auto-detect target linear modules for custom adapters
linear_layer_names = set()
for name, module in model.named_modules():
    if 'Linear' in type(module).__name__:
        linear_layer_names.add(name.split('.')[-1])

preferred = {'q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj',
             'query_key_value','dense','c_attn','c_proj','qkv_proj'}
target_modules = list(preferred & linear_layer_names) or list(linear_layer_names)[:6]
print(f"LoRA target modules: {target_modules}")

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=CONFIG['lora_r'],
    lora_alpha=CONFIG['lora_alpha'],
    target_modules=target_modules,
    lora_dropout=CONFIG['lora_dropout'],
    bias='none',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

ModuleNotFoundError: No module named 'peft'

In [ ]:
# ── Cell 8: Format dataset ────────────────────────────────────────────────────
from datasets import Dataset

def format_example(example):
    messages = example['messages']
    try:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    except Exception:
        text = ''.join(f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>\n" for m in messages)
    return {'text': text}

raw_ds = Dataset.from_list(records)
formatted = raw_ds.map(format_example, remove_columns=raw_ds.column_names)
split = formatted.train_test_split(test_size=CONFIG['val_split'], seed=42)

print(f"Train examples: {len(split['train'])} | Validation examples: {len(split['test'])}")
print('\nSample formatted prompt template (first 300 chars):')
print(split['train'][0]['text'][:300])

In [ ]:
# ── Cell 9: Train Model ───────────────────────────────────────────────────────
import os
from trl import SFTTrainer, SFTConfig

os.makedirs(CONFIG['output_dir'], exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
optim_choice = 'paged_adamw_8bit' if device == "cuda" else 'adamw_torch'
fp16_choice = True if device == "cuda" else False

# Handle Windows multiprocessing issue (dataloader workers set to 0 locally on Win)
num_workers = 0 if sys.platform.startswith('win') else 2

training_args = SFTConfig(
    output_dir=CONFIG['output_dir'],
    num_train_epochs=CONFIG['epochs'],
    per_device_train_batch_size=CONFIG['batch_size'],
    per_device_eval_batch_size=CONFIG['batch_size'],
    gradient_accumulation_steps=CONFIG['grad_accum'],
    learning_rate=CONFIG['lr'],
    weight_decay=0.01,
    warmup_ratio=0.05,
    lr_scheduler_type='cosine',
    optim=optim_choice,
    max_seq_length=CONFIG['max_seq_len'],
    logging_steps=10,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    load_best_model_at_end=True,
    fp16=fp16_choice,
    dataloader_num_workers=num_workers,
    report_to='none',
    dataset_text_field='text',
    remove_unused_columns=True,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=split['train'],
    eval_dataset=split['test'],
    processing_class=tokenizer,
)

print('Starting fine-tuning training loop...')
result = trainer.train()
print(f'Training complete! Final loss: {result.training_loss:.4f}')

In [ ]:
# ── Cell 10: Save Fine-Tuned Adapter ──────────────────────────────────────────
trainer.save_model(CONFIG['output_dir'])
tokenizer.save_pretrained(CONFIG['output_dir'])
print(f"Adapter weights successfully saved to: {CONFIG['output_dir']}")

import json, time
metrics = {
    'base_model': CONFIG['base_model'],
    'train_loss': round(result.training_loss, 4),
    'epochs': CONFIG['epochs'],
    'timestamp': time.strftime('%Y-%m-%dT%H:%M:%S'),
}
with open(os.path.join(CONFIG['output_dir'], 'training_metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics summary saved:', metrics)

In [ ]:
# ── Cell 11: Quick Inference Validation ───────────────────────────────────────
from peft import PeftModel

model.eval()
test_q = 'What are the early warning signs of liver disease?'
messages = [
    {'role': 'system', 'content': 'You are LiverAI, a liver health assistant.'},
    {'role': 'user',   'content': test_q},
]
try:
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
except Exception:
    input_text = f'<|im_start|>system\nYou are LiverAI.<|im_end|>\n<|im_start|>user\n{test_q}<|im_end|>\n<|im_start|>assistant\n'

inputs = tokenizer(input_text, return_tensors='pt').to(model.device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=250, temperature=0.3, do_sample=True,
                         pad_token_id=tokenizer.eos_token_id)

response = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(f"Q: {test_q}")
print(f"A: {response}")

In [ ]:
# ── Cell 12: Weight Packaging / Colab Export Helper ───────────────────────────
import shutil
import sys

is_colab = 'google.colab' in sys.modules

if is_colab:
    from google.colab import files
    zip_path = '/content/liver-lora.zip'
    shutil.make_archive('/content/liver-lora', 'zip', CONFIG['output_dir'])
    print(f"Zip archive created. Downloading: {zip_path}")
    files.download(zip_path)
    print("Done! Unzip into models/liver-lora/ in your local project directory.")
else:
    print(f"Running locally. Adapter weights saved directly in your workspace directory: {os.path.abspath(CONFIG['output_dir'])}")
    print("No zip/download required! Ensure this directory exists and is set in your .env file.")

## 🛠️ Post-Training Integration

1. **Weights Location**: Make sure the adapter weights are at `models/liver-lora/` (if trained locally, they will be there already).
2. **Edit `.env`** configuration:
   ```env
   USE_LOCAL_MODEL=true
   LOCAL_MODEL_PATH=models/liver-lora
   LOCAL_BASE_MODEL=microsoft/Phi-3-mini-4k-instruct
   ```
3. **Restart the server**: Run `python app.py` to automatically load and serve your fine-tuned local weights!